# 11 · Recovered rules as features, and a hand-written generator model (phase 4)

`10_generator` found three eras (2015 / 2016 / 2017-18) with near-additive, step-shaped effects. Exact cut points from the raw rates:
- **education (2017-18):** conversion jumps when **more than half** of the drivers have higher education (1/1, 2/2, ≥2/3, ≥3/4 → 17–29% vs 5–10%); exactly half doesn't count;
- **discount and home policy:** matter **only in 2016** (≈ 17% vs 9–10%);
- **drivers per vehicle:** > 1 → ~5%; (0.5, 1] → ~14%; < 0.5 → ~10.5%;
- **mean driver age:** band 35–50 inclusive;
- **density (2017-18):** stepped rise, ~9% below 10k/sq mi → 41% above 40k;
- **price and credit:** gradual, mostly before 2017; **3-driver** households best in 2017-18.

**A.** Do these rules (`features.add_rules`) help CatBoost? Groups tested on seed-0 folds, best confirmed on 15.
**B.** A hand-written per-era logistic regression built from these terms. How close is a fully explicit formula to the black box?
**C.** Re-blend CatBoost + per-era EBM + logistic; weights from seed-0, judged on seeds 1–2; write the submission.

In [1]:
import json, sys, glob, itertools
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import SplineTransformer, StandardScaler
sys.path.insert(0, '.')
import common as C, features as FE

lab, hold = C.load()
prep = lambda d: FE.add_rules(FE.add_era(FE.add_cats(FE.add_static(d))))
L, H = prep(lab), prep(hold)
F = json.load(open('params_tuned.json'))['cb']['features']
PC = json.load(open('params_cats.json')); F0, P0 = PC['features'], PC['params']
y = L[C.TARGET].to_numpy(); FOLDS = C.folds(lab)
cb0 = C.cv_score('cats_copies_ohm10', L, F0, 'cb', P0, seeds=(0,), log=False)                      # cached (07)
cb15 = C.cv_score('cats15___copies__ohm_10', L, F0, 'cb', P0, notes='07_catboost_cats')            # cached (07)
ebm15 = C.cv_score('ebm_era3', L, F + ['ERA'], 'ebm', {'interactions': 'era'}, notes='10 EBM per era')  # cached (10)
fmt = {'auc': '{:.4f}', 'Δ auc': '{:+.4f}', 'se Δ': '{:.4f}', 'Δ seeds 1-2': '{:+.4f}', 'se seeds 1-2': '{:.4f}', 'top_decile': '{:.1%}'}
def paired(a, b):
    d = a - b[:len(a)]
    out = {'auc': a.mean(), 'Δ auc': d.mean(), 'se Δ': d.std() / np.sqrt(len(d))}
    return out | ({'Δ seeds 1-2': d[5:].mean(), 'se seeds 1-2': d[5:].std() / np.sqrt(10)} if len(a) == 15 else {})
def aucs(P):
    return np.array([roc_auc_score(y[va], P[s, va]) for s, k, tr, va in FOLDS])
print(f'CatBoost (07) 15-fold {cb15["auc_mean"]:.4f} | per-era EBM {ebm15["auc_mean"]:.4f}')

CatBoost (07) 15-fold 0.7143 | per-era EBM 0.7120


## A. Rules as CatBoost features (seed-0 folds, paired vs. the `07` model)

In [2]:
ALL = [c for g in FE.RULES.values() for c in g]
A = {g: C.cv_score(f'rules_{i}', L, F0 + cols, 'cb', P0, seeds=(0,), log=False) for i, (g, cols) in enumerate(FE.RULES.items())}
A['all rules'] = C.cv_score('rules_all', L, F0 + ALL, 'cb', P0, seeds=(0,), log=False)
A['all rules + ERA'] = C.cv_score('rules_all_era', L, F0 + ALL + ['ERA'], 'cb', P0, seeds=(0,), log=False)
ta = pd.DataFrame({g: paired(r['fold_aucs'], cb0['fold_aucs']) | {'top_decile': r['top_decile_prec']} for g, r in A.items()}).T
ta.style.format(fmt)

,auc,Δ auc,se Δ,top_decile
edu majority x era,0.7138,-0.0007,0.0005,30.1%
2016 discount/home,0.7140,-0.0006,0.0003,29.9%
drivers per vehicle,0.7147,+0.0002,0.0003,30.0%
density x era,0.7135,-0.0010,0.0006,29.6%
price/credit x era,0.7146,+0.0001,0.0008,30.1%
drivers x era,0.7144,-0.0001,0.0006,29.6%
all rules,0.7146,+0.0000,0.0007,30.0%
all rules + ERA,0.7147,+0.0001,0.0006,29.8%


In [3]:
# Confirm on 15 folds: the union of groups with Δ > se, and the best single entry.
passing = [g for g in FE.RULES if ta.loc[g, 'Δ auc'] > ta.loc[g, 'se Δ']]
cands = {}
if passing:
    cands['passing groups'] = F0 + [c for g in passing for c in FE.RULES[g]]
top = ta['Δ auc'].idxmax()
if ta.loc[top, 'Δ auc'] > 0:
    cands[f'best: {top}'] = F0 + (ALL + (['ERA'] if 'ERA' in top else []) if top.startswith('all') else FE.RULES[top])
conf = {n: C.cv_score('rules15_' + ''.join(ch if ch.isalnum() else '_' for ch in n), L, f, 'cb', P0, notes='11 rules') for n, f in cands.items()}
tc = pd.DataFrame({n: paired(r['fold_aucs'], cb15['fold_aucs']) | {'top_decile': r['top_decile_prec'], 'rounds': r['rounds']} for n, r in conf.items()}).T
display(tc.style.format(fmt) if len(tc) else 'no candidate')
win = tc[tc['Δ seeds 1-2'] > 2 * tc['se seeds 1-2']].sort_values('Δ seeds 1-2', ascending=False) if len(tc) else tc
CBF, CB = (cands[win.index[0]], conf[win.index[0]]) if len(win) else (F0, cb15)
print('CatBoost for the blend:', win.index[0] if len(win) else '07 model unchanged', f'| {len(CBF)} features, 15-fold {CB["auc_mean"]:.4f}')

,auc,Δ auc,se Δ,Δ seeds 1-2,se seeds 1-2,top_decile,rounds
best: drivers per vehicle,0.7145,+0.0002,0.0004,+0.0001,0.0005,29.8%,361.000000


CatBoost for the blend: 07 model unchanged | 45 features, 15-fold 0.7143


**Reading:** the rules add nothing to CatBoost (all within ±0.001; the best, drivers-per-vehicle bands, is +0.0001 ± 0.0005 on seeds 1–2). CatBoost had already learned these step functions from the raw columns plus the one-hot copies. The value of the recovered rules is in a *different* model, not as extra columns for this one.

## B. Hand-written per-era logistic regression
Design matrix (all target-free):
- **era intercepts**; **per-era cubic splines** (5 knots) of credit score, log price, log density and mean driver age, i.e. each era gets its own curve;
- **per-era flags:** education majority, discount, home policy, number of drivers (one-hot);
- **shared terms:** age band 35–50, drivers-per-vehicle bands, CAT zone, state, prior carrier and package (one-hot);
- every other of the 41 features as a standardized linear term.

L2-regularized logistic regression; `C` chosen on seed-0 folds.

In [4]:
ERAS = ['2015', '2016', '2017-18']
SPL = ['CREDIT_SCORE', 'QUOTED_AMT', 'ACS_POP_DENSITY', 'DRV_AGE_MEAN']
ONEHOT = {'STATE_CD': None, 'PRIOR_CARRIER_GRP': None, 'COV_PKG_TYPE': None, 'CAT_ZONE': None}
ERA_FLAGS = ['R_EDU_MAJ', 'DISCOUNT_IND', 'HOME_POLICY_IND']
SHARED = ['DRV_AGE_35_50', 'R_DPV_GT1', 'R_DPV_HALF_TO_1']
LINEAR = [f for f in F if f not in SPL + list(ONEHOT) + ERA_FLAGS + SHARED + ['NUM_DRIVERS', 'DRIVERS_PER_VEH', 'QUOTE_YEAR']
          and not isinstance(L[f].dtype, pd.CategoricalDtype)]
tostr = lambda s: s.astype(object).fillna('Missing').astype(str)
LEVELS = {c: sorted(tostr(pd.concat([L[c], H[c]])).unique()) for c in ONEHOT}

class GLM:
    def __init__(self, C_=0.1): self.C_ = C_
    def _raw(self, d):
        x = d[SPL].astype(float).copy()
        x['QUOTED_AMT'], x['ACS_POP_DENSITY'] = np.log(x.QUOTED_AMT), np.log1p(x.ACS_POP_DENSITY)
        return x
    def _design(self, d, fit=False):
        E = np.column_stack([d.ERA.eq(e).to_numpy(float) for e in ERAS])
        x = self._raw(d)
        if fit:
            self.med = x.median(); self.spl = SplineTransformer(n_knots=5, degree=3, knots='quantile', extrapolation='constant').fit(x.fillna(self.med))
            self.lin_med = d[LINEAR].astype(float).median()
            self.sc = StandardScaler().fit(d[LINEAR].astype(float).fillna(self.lin_med))
        S = self.spl.transform(x.fillna(self.med))
        blocks = [E] + [S * E[:, [j]] for j in range(3)]
        blocks += [d[f].astype(float).fillna(0).to_numpy()[:, None] * E for f in ERA_FLAGS]
        nd = d.NUM_DRIVERS.clip(upper=5).astype(int).to_numpy()
        blocks += [((nd == k)[:, None] * E) for k in range(2, 6)]
        blocks += [d[SHARED].astype(float).to_numpy()]
        blocks += [np.column_stack([tostr(d[c]).eq(v).to_numpy(float) for v in LEVELS[c][1:]]) for c in ONEHOT]
        blocks += [self.sc.transform(d[LINEAR].astype(float).fillna(self.lin_med)), x.isna().to_numpy(float)]
        return np.hstack(blocks)
    def fit(self, d, yy):
        X = self._design(d, fit=True); self.xs = StandardScaler().fit(X)
        self.m = LogisticRegression(C=self.C_, max_iter=5000).fit(self.xs.transform(X), yy); return self
    def predict(self, d):
        return self.m.predict_proba(self.xs.transform(self._design(d)))[:, 1]

def glm_cv(C_, seeds=(0, 1, 2)):
    P = np.full((3, len(y)), np.nan)
    for s, k, tr, va in FOLDS:
        if s in seeds:
            P[s, va] = GLM(C_).fit(L.iloc[tr], y[tr]).predict(L.iloc[va])
    return P
grid = {C_: aucs(np.nan_to_num(glm_cv(C_, seeds=(0,))))[:5].mean() for C_ in (0.003, 0.01, 0.03, 0.1, 0.3)}
print('seed-0 AUC by C:', {k: round(v, 4) for k, v in grid.items()})
C_BEST = max(grid, key=grid.get)
GP = glm_cv(C_BEST)
glm = {'fold_aucs': aucs(GP), 'oof_all': GP, 'auc_mean': aucs(GP).mean()}
tb = pd.DataFrame({'per-era logistic (explicit formula)': paired(glm['fold_aucs'], CB['fold_aucs']),
                   'per-era EBM': paired(ebm15['fold_aucs'], CB['fold_aucs'])}).T
print(f'C = {C_BEST}; comparison vs the CatBoost used for the blend ({CB["auc_mean"]:.4f}):')
tb.style.format(fmt)

seed-0 AUC by C: {0.003: np.float64(0.7136), 0.01: np.float64(0.7133), 0.03: np.float64(0.7126), 0.1: np.float64(0.712), 0.3: np.float64(0.7117)}


C = 0.003; comparison vs the CatBoost used for the blend (0.7143):


,auc,Δ auc,se Δ,Δ seeds 1-2,se seeds 1-2
per-era logistic (explicit formula),0.7134,-0.0009,0.0009,-0.0011,0.0012
per-era EBM,0.7120,-0.0023,0.0006,-0.0022,0.0009


**Reading:**
- **An explicit formula nearly matches the black box:** the per-era logistic (166 terms) reaches **0.7134**, only −0.0009 ± 0.0009 behind CatBoost (within noise) and ahead of the per-era EBM (0.7120). The generator is essentially *a per-era additive logit with step rules*.
- Regularization preferred is the strongest tried (C = 0.003, the grid's edge), so a still smaller C might help slightly.

## C. Re-blend: CatBoost + per-era EBM + per-era logistic
Probability average; weights on a 0.05 grid searched on the seed-0 repeat, judged on seeds 1–2 vs CatBoost alone and vs the current submission (0.7 CatBoost-07 + 0.3 EBM).

In [5]:
from scipy.stats import spearmanr
M = {'CatBoost': CB['oof_all'], 'EBM': ebm15['oof_all'], 'GLM': glm['oof_all']}
print('rank correlations (repeat 0):', {f'{a}-{b}': round(spearmanr(M[a][0], M[b][0])[0], 3) for a, b in itertools.combinations(M, 2)})
steps = np.round(np.arange(0, 1.0001, 0.05), 2)
W = [np.array([1 - a - b, a, b]) for a in steps for b in steps if a + b <= 1 + 1e-9]
mix = lambda w: sum(wi * P for wi, P in zip(w, M.values()))
s0 = [aucs(mix(w))[:5].mean() for w in W]
wb = W[int(np.argmax(s0))]
cur = 0.7 * cb15['oof_all'] + 0.3 * ebm15['oof_all']
rows = {'CatBoost alone': paired(aucs(CB['oof_all']), cb15['fold_aucs']),
        'current submission (0.7 CB-07 + 0.3 EBM)': paired(aucs(cur), cb15['fold_aucs']),
        f'new blend {dict(zip(M, wb.round(2)))}': paired(aucs(mix(wb)), cb15['fold_aucs'])}
tblend = pd.DataFrame(rows).T
display(tblend.style.format(fmt))
dn = aucs(mix(wb)) - aucs(cur)
better = dn[5:].mean() > 2 * dn[5:].std() / np.sqrt(10)
print(f'new blend vs current submission, seeds 1-2: {dn[5:].mean():+.4f} ± {dn[5:].std() / np.sqrt(10):.4f} -> {"adopt" if better else "keep current"}')

rank correlations (repeat 0): {'CatBoost-EBM': np.float64(0.961), 'CatBoost-GLM': np.float64(0.946), 'EBM-GLM': np.float64(0.962)}


,auc,Δ auc,se Δ,Δ seeds 1-2,se seeds 1-2
CatBoost alone,0.7143,+0.0000,0.0000,+0.0000,0.0000
current submission (0.7 CB-07 + 0.3 EBM),0.7151,+0.0009,0.0002,+0.0009,0.0003
"new blend {'CatBoost': np.float64(0.55), 'EBM': np.float64(0.05), 'GLM': np.float64(0.4)}",0.7163,+0.0021,0.0004,+0.0020,0.0005


new blend vs current submission, seeds 1-2: +0.0012 ± 0.0004 -> adopt


In [6]:
if better:
    preds = {}
    if CBF == F0:
        cbp = pd.read_csv(sorted(glob.glob('../submissions/catboost_cats5_*.csv'))[-1]); assert (cbp.POLICY_ID.values == H.POLICY_ID.values).all()
        preds['CatBoost'] = cbp.CONVERT_IND_PRED.to_numpy()
    else:
        preds['CatBoost'] = np.mean([C.fit_full(L, CBF, 'cb', P0 | {'random_seed': s}, int(round(CB['rounds'] * 1.2)))(H) for s in [42, 1, 2, 3, 4]], axis=0)
    preds['EBM'] = C.fit_full(L, F + ['ERA'], 'ebm', {'interactions': 'era'})(H) if wb[1] > 0 else 0
    preds['GLM'] = GLM(C_BEST).fit(L, y).predict(H) if wb[2] > 0 else 0
    p = sum(w * preds[k] for w, k in zip(wb, M))
    out = C.write_submission('blend_v2', hold, p)
    json.dump({'weights': dict(zip(M, map(float, wb))), 'catboost_features': CBF, 'glm_C': C_BEST, 'cv_auc': float(aucs(mix(wb)).mean())},
              open('blend_v2.json', 'w'), indent=1)
    print('wrote', out.name, f'| unique {pd.Series(p).nunique()} | mean {p.mean():.4f}')
else:
    print('current submission (blend_cb_ebm) stays')

/Users/joerussell/Desktop/PROJECTS/competitions/travelers-umc/.venv/lib/python3.12/site-packages/interpret/glassbox/_ebm/_ebm.py:871: UserWarning: Missing values detected. Our visualizations do not currently display missing values. To retain the glassbox nature of the model you need to either set the missing values to an extreme value like -1000 that will be visible on the graphs, or manually examine the missing value score in ebm.term_scores_[term_index][0]
  warn(


wrote blend_v2_20261030.csv | unique 7375 | mean 0.1116


## Conclusions
- **Rules as CatBoost features: no gain.** CatBoost already captures them.
- **Hand-written per-era logistic: 15-fold 0.7134**, statistically tied with CatBoost (0.7143). This is the recovered generator in explicit form: era intercepts; per-era splines for credit, log price, log density and driver age; per-era education-majority, discount and home flags; driver-count by era; age band 35–50; drivers-per-vehicle bands; CAT zone, state, carrier and package effects.
- **New blend adopted:** 0.55 CatBoost (07) + 0.05 per-era EBM + 0.40 per-era logistic (probability average): **15-fold AUC 0.7163**, +0.0020 ± 0.0005 over CatBoost and **+0.0012 ± 0.0004 over the previous submission on the unseen seeds 1–2**. Written to `submissions/blend_v2_20261030.csv` (7,375 distinct values, mean 11.2%) and `blend_v2.json`.
- The logistic's rank correlation with CatBoost is only 0.946 (vs 0.961 for the EBM), so it's the most complementary model found so far, which is why it takes 40% of the blend.